# Whole-corpus Word2Vec (backup model)

A single Word2Vec model trained on the *entire* corpus, not split by era -- a backup /
supplement to `SF_word2vec_eras_v2.ipynb`, which stays the primary tool for the actual
diachronic (era-vs-era) analysis. This model exists for cases where more data per word
matters more than era resolution: general nearest-neighbor sanity checks, coverage for
words too rare within any single era to trust, or a fallback if an era-specific finding
needs a whole-corpus comparison point.

Trained on the same audited htid set as the era models (`metadata_august2026.csv`, via
`load_metadata()`'s Ace-Doubles-aware grouping), so it's a fair comparison against them --
just not split into `era_a`/`era_b`/`era_c`.

Multi-seed (same as the era models) so neighbor-list findings can still be checked for
stability rather than trusted from a single run.

This notebook also handles releasing the trained model files themselves (not just
aggregate tables) so they can be reloaded and used entirely outside the capsule, the same
way `era_a`/`era_b`/`era_c`'s models already were -- raw trained-model files have been
accepted by HTRC review before (per Dez, "they'll accept it, they accepted one before"),
but expect this to need manual review rather than auto-approval, same as that release --
these are large binary files (~150-300MB+ per seed), not small aggregate CSVs.

Must run **inside the capsule, in secure mode**. Word2Vec training itself is CPU-bound in
gensim (no GPU acceleration), so this runs on the standard capsule, not the GPU box.

## Imports

In [ ]:
import json
import os
import re
from collections import Counter
from multiprocessing import get_context
from pathlib import Path

import pandas as pd
from gensim.models import Word2Vec
from gensim.models.phrases import Phrases, Phraser
from nltk.tokenize import TreebankWordTokenizer, sent_tokenize

## Check the sentence tokenizer is available

Secure mode has no network -- if this fails, run
`python3 -c "import nltk; nltk.download('punkt'); nltk.download('punkt_tab')"` from
maintenance mode first, then switch to secure mode and retry.

In [ ]:
try:
    sent_tokenize("Checking that the sentence tokenizer's data is available. This is only a test.")
    print("tokenizer OK")
except LookupError as e:
    raise SystemExit(
        "NLTK sentence-tokenizer data not found, and secure mode has no network to fetch it. "
        "Run this from maintenance mode BEFORE switching to secure mode:\n"
        "  python3 -c \"import nltk; nltk.download('punkt'); nltk.download('punkt_tab')\"\n"
        f"Original error: {e}"
    )

## OCR cleaning

Copied verbatim from `SF_word2vec_eras_v2.ipynb` (cell `46a9ca82`).

In [ ]:
_norm_ws = re.compile(r"\s+")
_only_number = re.compile(r"^\s*[\divxlcIVXLC]+\s*$")
_hyphen_break = re.compile(r"(\w)-\s*\n\s*(\w)")
_word = re.compile(r"[A-Za-z']+")


def discover_volumes(base_dir, ids=None):
    vols = {}
    for d in sorted(p for p in Path(base_dir).iterdir() if p.is_dir() and not p.name.startswith(".")):
        if ids is not None and d.name not in ids:
            continue
        pages = sorted(d.glob("*.txt"))
        if pages:
            vols[d.name] = pages
    return vols


def _norm_line(line):
    return _norm_ws.sub(" ", re.sub(r"\d+", "", line)).strip().lower()


def _volume_vocab(page_paths):
    words = set()
    for p in page_paths:
        text, _ = _hyphen_break.subn(r"\1\2", p.read_text(encoding="utf-8", errors="replace"))
        words.update(w.lower() for w in _word.findall(text) if len(w) > 1)
    return words


def build_dictionary(volumes, min_vols, procs=None):
    df = Counter()
    with get_context("fork").Pool(procs or min(16, os.cpu_count() or 1)) as pool:
        for vocab in pool.imap_unordered(_volume_vocab, list(volumes.values()), chunksize=8):
            df.update(vocab)
    return {w for w, c in df.items() if c >= min_vols}


def page_dict_rate(text, dictionary):
    words = [w.lower() for w in _word.findall(text) if len(w) > 1]
    return sum(1 for w in words if w in dictionary) / len(words) if words else 0.0


def clean_volume(page_paths, dictionary, running_head_min_pages, running_head_frac,
                  running_head_max_chars, page_min_dict_rate):
    pages = [p.read_text(encoding="utf-8", errors="replace") for p in page_paths]
    line_pages = Counter()
    per_page_lines = []
    for text in pages:
        lines = text.split("\n")
        per_page_lines.append(lines)
        line_pages.update({_norm_line(l) for l in lines if 0 < len(l.strip()) <= running_head_max_chars})
    thresh = max(running_head_min_pages, int(running_head_frac * len(pages)))
    heads = {l for l, c in line_pages.items() if c >= thresh and l}
    kept = []
    for lines in per_page_lines:
        out = []
        for l in lines:
            if (len(l.strip()) <= running_head_max_chars and _norm_line(l) in heads) or _only_number.match(l):
                continue
            out.append(l)
        page_text, _ = _hyphen_break.subn(r"\1\2", "\n".join(out))
        if dictionary is not None and page_min_dict_rate and page_dict_rate(page_text, dictionary) < page_min_dict_rate:
            continue
        kept.append(page_text)
    text, _ = _hyphen_break.subn(r"\1\2", "\n".join(kept))
    return text


def load_docs(base_dir, dictionary, ids=None, **clean_kwargs):
    docs = []
    for htid, pages in discover_volumes(base_dir, ids).items():
        docs.append(clean_volume(pages, dictionary, **clean_kwargs))
    return docs


_tokenizer = TreebankWordTokenizer()
TOKEN_RE = re.compile(r"^[a-z]+(?:'[a-z]+)?$")


def clean_tokens(tokens):
    return [t for t in tokens if TOKEN_RE.match(t) and len(t) > 1]


def make_sentences(list_text):
    """Lowercase, sentence-split, word-tokenize, then strip OCR noise from each sentence."""
    all_sentences = []
    for txt in list_text:
        lower_txt = txt.lower()
        for sent in sent_tokenize(lower_txt):
            cleaned = clean_tokens(_tokenizer.tokenize(sent))
            if cleaned:
                all_sentences.append(cleaned)
    return all_sentences

## Metadata loading

Only used here to get the same audited htid set as the era models train on (same
Ace-Doubles-aware grouping as `SF_word2vec_eras_v2.ipynb` / `SF_novel_keyness_v2.ipynb`) --
year/title/author aren't otherwise needed since this model isn't split by era.

In [ ]:
def load_metadata(path):
    df = pd.read_csv(path)
    df["htid"] = df["htid"].astype(str)

    def join_unique(values):
        return " / ".join(dict.fromkeys(str(v) for v in values))

    grouped = df.groupby("htid").agg(
        title=("title", join_unique),
        author=("author", join_unique),
        year=("year", "first"),
    )
    return grouped.to_dict("index")

## Config

In [ ]:
TEXT_DIR = "/media/secure_volume/fa50b375-3216-4edd-a685-98488562b723"
METADATA_CSV = "/home/dcuser/Desktop/Clifi-htrc/notebooks/metadata_august2026.csv"
MODEL_DIR = Path("/media/secure_volume/whole_corpus_word2vec_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DICT_MIN_VOLS = 10
PAGE_MIN_DICT_RATE = 0.55
RUNNING_HEAD_MIN_PAGES = 3
RUNNING_HEAD_FRAC = 0.05
RUNNING_HEAD_MAX_CHARS = 60

W2V_PARAMS = dict(
    min_count=2,
    vector_size=300,
    sg=1,   # skip-gram -- same as the era models
    workers=min(os.cpu_count() or 4, 12),
)
SEEDS = [1, 2, 3]   # same multi-seed convention as the era models, for a stability check

## Load metadata

In [ ]:
meta_by_id = load_metadata(METADATA_CSV)
valid_ids = set(meta_by_id.keys())
print(f"{len(valid_ids):,} unique htids in the audited metadata")

## Discover volumes + build corpus dictionary

In [ ]:
all_volumes = discover_volumes(TEXT_DIR, ids=valid_ids)
print(f"found {len(all_volumes):,} volumes (of {len(valid_ids):,} in metadata)")

dictionary = build_dictionary(all_volumes, DICT_MIN_VOLS)
print(f"dictionary: {len(dictionary):,} words appear in >= {DICT_MIN_VOLS} volumes")

## Clean + load every volume, then sentence-split -- THE SLOW STEP

One combined sentence list for the whole corpus (no era split). Once this finishes,
`sentences` stays in kernel memory -- the training step below can be re-run without
paying this cost again.

In [ ]:
docs = load_docs(
    TEXT_DIR, dictionary, ids=valid_ids,
    running_head_min_pages=RUNNING_HEAD_MIN_PAGES,
    running_head_frac=RUNNING_HEAD_FRAC,
    running_head_max_chars=RUNNING_HEAD_MAX_CHARS,
    page_min_dict_rate=PAGE_MIN_DICT_RATE,
)
print(f"loaded {len(docs):,} cleaned documents")

sentences = make_sentences(docs)
print(f"{len(sentences):,} sentences")

## Bigram phrase detection

Fit on the whole corpus (there's only one corpus here, unlike the era notebook, which has
to fit on the combined eras specifically so phrase tokens like `acid_rain` stay consistent
across era comparisons -- not a concern here).

In [ ]:
phrases = Phrases(sentences, min_count=5, threshold=10)
phraser = Phraser(phrases)
phrased_sentences = [phraser[s] for s in sentences]
print("phrase detection done")

## Train Word2Vec (multiple seeds for stability)

Safe to re-run this cell alone if a later step fails -- `phrased_sentences` is already in
memory from the cells above.

In [ ]:
models = {}
for seed in SEEDS:
    print(f"training seed {seed} ...")
    m = Word2Vec(phrased_sentences, seed=seed, **W2V_PARAMS)
    models[seed] = m
    print(f"  seed {seed}: vocab {len(m.wv.key_to_index):,}")

## Save the models

Safe to re-run this cell (and the verify cell after it) as many times as needed --
`models` is already in memory.

In [ ]:
model_paths = {}
for seed, m in models.items():
    path = MODEL_DIR / f"sf_model_whole_corpus_seed{seed}"
    m.save(str(path))
    model_paths[seed] = path
    print(f"wrote {path}")

## Verify every model actually landed on disk

Re-loads each model fresh from disk (not from the `models` variable already in memory)
and checks its vocab size matches -- this is the check that a previous run of a different
notebook skipped, and it cost a multi-hour redo when a write silently didn't persist.
`Word2Vec.load()` also implicitly checks that every expected `.npy` companion file
(`.wv.vectors.npy`, `.syn1neg.npy`) is actually present, not just the main pickle.

In [ ]:
for seed, path in model_paths.items():
    reloaded = Word2Vec.load(str(path))
    expected_vocab = len(models[seed].wv.key_to_index)
    actual_vocab = len(reloaded.wv.key_to_index)
    assert actual_vocab == expected_vocab, (
        f"MISMATCH for seed {seed}: trained model has {expected_vocab} vocab, "
        f"but reloading {path} from disk shows {actual_vocab}"
    )
    print(f"VERIFIED: seed {seed} reloads from disk with {actual_vocab:,} vocab, matches expected.")

## Sanity check

Not a substitute for real analysis -- just eyeballing that training produced sensible
neighbors before trusting the release.

In [ ]:
sanity_words = ["climate", "disaster", "nuclear", "pollution", "war", "earth"]
primary = models[SEEDS[0]]
for word in sanity_words:
    if word not in primary.wv.key_to_index:
        print(f"'{word}': not in vocabulary")
        continue
    neighbors = primary.wv.most_similar(word, topn=6)
    print(f"'{word}': {[w for w, _ in neighbors]}")

## Release the trained model files

Every seed's model (3 files each: the main pickle, `.wv.vectors.npy`, `.syn1neg.npy`) --
raw vectors, not text. Expect this to need manual HTRC review rather than auto-approval,
same as the earlier era-model release did (these are large binary files, not small
aggregate tables). `add` and `done` are kept in separate cells so you can review what
got queued before finalizing.

In [ ]:
import subprocess


def run_releaseresults(*args):
    cmd = ["releaseresults", *args]
    print("$ " + " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"releaseresults exited with code {result.returncode}")
    return result

In [ ]:
model_files = sorted(str(p) for p in MODEL_DIR.iterdir() if p.is_file())
print(f"{len(model_files)} files to release:")
for f in model_files:
    print(" ", f)
run_releaseresults("add", *model_files)

Check the output above looks right, then run this to finalize:

In [ ]:
run_releaseresults("done")